# Legacy Code Modernization Benchmark: Kaggle Quickstart

Load the public tasks, inspect source programs, add translations, and write a Kaggle-compatible CSV. This notebook does not execute translations or reveal hidden evaluation cases.

In [ ]:
import csv
import json
from pathlib import Path

def find_file(filename):
    roots = [Path('/kaggle/input'), Path('kaggle'), Path('../kaggle')]
    for root in roots:
        if root.exists():
            matches = list(root.rglob(filename)) if root.is_dir() else []
            if matches:
                return matches[0]
    raise FileNotFoundError(f'Could not find {filename}. Attach the benchmark dataset on Kaggle or run from the repository root.')

tasks_path = find_file('tasks.jsonl')
tasks = [json.loads(line) for line in tasks_path.read_text(encoding='utf-8').splitlines() if line.strip()]
template_path = find_file('sample_submission.csv')
with template_path.open(newline='', encoding='utf-8') as template_file:
    submission_template = list(csv.DictReader(template_file))
if {row['task_id'] for row in submission_template} != {task['task_id'] for task in tasks}:
    raise ValueError('The sample submission IDs do not match the public task IDs.')
print(f'Loaded {len(tasks)} tasks from {tasks_path}')
print(f'Loaded submission template from {template_path}')
print('Task pairs:', sorted({(task["source_language"], task["target_language"]) for task in tasks}))

In [ ]:
for task in tasks:
    print(f'\n--- {task["task_id"]}: {task["source_language"]} -> {task["target_language"]} ---')
    print(f'Family: {task["family_id"]} | Difficulty: {task["difficulty"]}/5')
    print(f'Legacy features: {", ".join(task["legacy_features"])}')
    print(f'Reference runtime: {task["reference_runtime"]}')
    print(f'Provenance: {task["provenance"]} | Version: {task["version"]}')
    print(task['prompt'])
    print(task['source_code'])

## Add translations

Enter one complete target-language program per task. Each program should read from standard input and write its answer to standard output. The starter values below are intentionally empty; fill them before creating a scored submission.

In [ ]:
translations = {row['task_id']: '' for row in submission_template}

# Example: translations['some_task_id'] = 'print(42)\n'
# Replace each empty value above with a complete translation.

In [ ]:
missing = [task_id for task_id, code in translations.items() if not code.strip()]
if missing:
    raise ValueError(f'Add translations for all tasks before submitting. Missing: {missing}')

output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission.csv'
with submission_path.open('w', newline='', encoding='utf-8') as submission_file:
    writer = csv.writer(submission_file)
    writer.writerow(['task_id', 'translated_code'])
    writer.writerows((task_id, translations[task_id]) for task_id in translations)
print(f'Wrote {submission_path}')

## Evaluation and safety

This notebook only packages translations. It does not execute submitted programs, score hidden tests, or provide a leaderboard. For a hosted competition, keep expected outputs private and use an approved, isolated evaluator; the repository's local scorer is for trusted code only.